## Using STED-FM model in PhenoMe pipeline

### 0. Setup

In [204]:
import os
import torch
import numpy as np
import pandas as pd
from stedfm import get_pretrained_model_v2
from phenome.utils import ModelWrapper
from phenome import PhenoMe, load_dinov2_model

# Initialize PhenoMe object
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pheno = PhenoMe(device=device, seed=42)

#### Choose dataset and model to use

In [ ]:
# Choose one of 3 datasets from ("NAS", "Optim", "Zooniverse")
DATASET_NAME = "NAS"
#DATASET_NAME = "Optim"
#DATASET_NAME = "Zooniverse"

# Choose one of 2 models from ("STED-FM", "DinoV2")
MODEL = "STED-FM"
#MODEL = "DinoV2"

#### Download and setup data

3 STED datasets are available: Zooniverse, NAS and Optim

In [206]:
# Decide where to save datasets
DATASETS_PATH = "stedfm_datasets"
os.makedirs(DATASETS_PATH, exist_ok=True)

DATA_DIR = os.path.join(DATASETS_PATH, f"{DATASET_NAME}_dataset")
os.makedirs(DATA_DIR, exist_ok=True)

In [207]:
from etl import download_dataset, transform_dataset

# Download and transform dataset to fit PhenoMe's required structure
download_dataset(DATASET_NAME, DATA_DIR)
transform_dataset(DATASET_NAME, DATA_DIR, DATA_DIR)

Zoo dataset already exists!
Done!
Transforming Zooniverse dataset...
Zoo dataset already transformed!
Done!


### 1. Wrap the STED-FM pretrained model

In [208]:
class STEDFMWrapper(ModelWrapper):
    def __init__(self, device):
        # Load the pretrained STED-FM model
        model, cfg = get_pretrained_model_v2(
            name = "mae-lightning-small",
            weights = "MAE_SMALL_STED", 
            in_channels=1,
            as_classifier=True,
        )

        # Initialize the parent class which moves the model to the device and sets it to eval()
        super().__init__(model=model, device=device)

    def _get_embeddings(self, tensor: torch.Tensor) -> torch.Tensor:
        out = self.model.forward_features(tensor)
        return out

if MODEL == "STED-FM":
    wrapper = STEDFMWrapper(device=device)
elif MODEL == "DinoV2":
    wrapper = load_dinov2_model(model_name="dinov2_vits14_reg", device=device)

in_channels 1
--- mae-lightning-small | (https://s3.valeria.science/flclab-foundation-models/models/mae-small-sted.zip) Downloading from URL and extracting zip ---

Loading state_dict from /Users/renaud/.stedfm/baselines/mae-small_STED/pl_checkpoint-999.pth
--- Loaded model mae-lightning-small with weights MAE_SMALL_STED ---
--- Added linear probe to all frozen blocks ---


### 2. Use custom transforms

The default PhenoMe transforms apply standardisation based on ImageNet values, which isn't appropriate for the STED-FM model.
The STED datasets used are already normalized for STED-FM usage.

In [209]:
import torchvision.transforms as T

TARGET_SIZE = 224

if MODEL == "STED-FM":
    # Transforms for STED-FM model
    custom_transforms = T.Compose([
        T.ToTensor(),
        T.Resize((TARGET_SIZE, TARGET_SIZE), interpolation=T.InterpolationMode.NEAREST),
    ])
elif MODEL == "DinoV2":
    # Transforms for DinoV2 model
    custom_transforms = T.Compose([
        T.ToTensor(),
        T.Resize((TARGET_SIZE, TARGET_SIZE), interpolation=T.InterpolationMode.NEAREST),
        T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
    ])

### 3. Load data (find_files)

We assume CSV-based metadata. The STED datasets were already normalized. If raw data is used, it should be normalized based on percentile values from **entire images**, not by crop.

In [210]:

from phenome import make_dataframe_metadata_fn
import pandas as pd

images_dir = os.path.join(DATA_DIR, "images")
masks_dir = os.path.join(DATA_DIR, "masks") # Optional
metadata_dir = os.path.join(DATA_DIR, "metadata.csv")

metadata_df = pd.read_csv(metadata_dir)
metadata_fn = make_dataframe_metadata_fn(metadata_df)
file_df = pheno.find_files(images_dir, mask_dir=masks_dir, metadata_fn=metadata_fn)

print(f"Found {len(file_df)} images")

Found 1084 files total.
Matched masks: 1084/1084 images (0 missing).
DataFrame: 1084 files, 33 columns.


Found 1084 images


### 4. Process images

In [211]:
# Base path to outputs: checkpoints and plots
OUTPUT_PATH = "output"
os.makedirs(OUTPUT_PATH, exist_ok=True)

CHECKPOINTS_PATH = os.path.join(OUTPUT_PATH, "checkpoints")
os.makedirs(CHECKPOINTS_PATH, exist_ok=True)

In [212]:
checkpoint_path = os.path.join(CHECKPOINTS_PATH, DATASET_NAME + f"_{MODEL}" + "_checkpoint.hdf5")

if MODEL == "STED-FM":
    pheno.process_images(wrapper, 
                        force_rgb=False,  # STED-FM model expects a single channel
                        checkpoint_path=checkpoint_path,
                        custom_transformations=custom_transforms,
                        )
elif MODEL == "DinoV2":
    pheno.process_images(wrapper, 
                    force_rgb=True,
                    channel_mode='combined',
                    checkpoint_path=checkpoint_path,
                    custom_transformations=custom_transforms,
                    )
pheno.compute_properties(property_preset="complete", checkpoint_path=checkpoint_path)

Found 1084 committed images; skipping 1084 (already in checkpoint), 0 to process.
All images already processed. Loading from checkpoint.
Pipeline ready: 1084 images available (lazy loading from output/checkpoints/Zooniverse_STED-FM_checkpoint.hdf5).
Masks: 1084/1084 images.
Properties: 1084/1084 matched in checkpoint (32 columns).


,image_index,image_path,image_name,area,circularity,eccentricity,equivalent_diameter,euler_number,extent,intensity_entropy,...,ring_2_std,ring_3_mean,ring_3_std,sharpness_metric,solidity,texture_contrast,texture_correlation,texture_dissimilarity,texture_energy,texture_homogeneity
0,0,stedfm_datasets/Zooniverse_dataset/images/R-Ho...,R-Homer_GARSTAR635_M-PSD95_GAMAlexa594_GluGly_...,434.0,0.297782,0.925953,23.507147,2.0,0.344992,7.040886,...,0.132237,0.762327,0.164634,0.477658,0.698873,767.112488,0.902193,12.849445,0.602411,0.620878
1,1,stedfm_datasets/Zooniverse_dataset/images/R-Ho...,R-Homer_GARSTAR635_M-PSD95_GAMAlexa594_GluGly_...,380.0,0.386492,0.948557,21.996159,1.0,0.400000,6.498264,...,0.095242,0.492432,0.089146,0.495540,0.727969,440.471710,0.898478,11.029917,0.543413,0.563892
2,2,stedfm_datasets/Zooniverse_dataset/images/R-Ho...,R-Homer_GARSTAR635_M-PSD95_GAMAlexa594_GluGly_...,2153.0,0.251407,0.890856,52.357281,0.0,0.441189,7.529780,...,0.183015,0.626849,0.171771,0.505496,0.648299,811.364014,0.937135,13.356783,0.530597,0.561250
3,3,stedfm_datasets/Zooniverse_dataset/images/R-Ho...,R-Homer_GARSTAR635_M-PSD95_GAMAlexa594_GluGly_...,250.0,0.622994,0.943404,17.841242,1.0,0.480769,6.975996,...,0.129222,0.796850,0.108634,0.576560,0.880282,1133.115112,0.911773,19.801729,0.460333,0.475545
4,4,stedfm_datasets/Zooniverse_dataset/images/R-Ho...,R-Homer_GARSTAR635_M-PSD95_GAMAlexa594_GluGly_...,270.0,0.409614,0.968693,18.541162,1.0,0.441176,6.760745,...,0.203385,0.990872,0.047307,0.493635,0.752089,2131.103271,0.890132,23.650383,0.490157,0.577176
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1079,1079,stedfm_datasets/Zooniverse_dataset/images/R-Ri...,R-Rim_GARSTAR635-M-Bassoon_GAMAlexa594_0MgGlyB...,210.0,0.983528,0.556693,16.351768,1.0,0.823529,6.281836,...,0.038156,0.291709,0.043266,0.495094,0.967742,196.952774,0.821496,10.140196,0.107486,0.181099
1080,1080,stedfm_datasets/Zooniverse_dataset/images/R-Ri...,R-Rim_GARSTAR635-M-Bassoon_GAMAlexa594_0MgGlyB...,516.0,0.757879,0.779905,25.631847,1.0,0.613555,7.899512,...,0.136390,0.918172,0.090355,0.707125,0.902098,1039.812622,0.935577,17.538837,0.326567,0.379459
1081,1081,stedfm_datasets/Zooniverse_dataset/images/R-Ri...,R-Rim_GARSTAR635-M-Bassoon_GAMAlexa594_0MgGlyB...,295.0,0.852549,0.378156,19.380548,1.0,0.739348,5.695748,...,0.025256,0.171198,0.020361,0.458195,0.936508,102.122849,0.780238,7.001076,0.169424,0.272820
1082,1082,stedfm_datasets/Zooniverse_dataset/images/R-Ri...,R-Rim_GARSTAR635-M-Bassoon_GAMAlexa594_0MgGlyB...,184.0,0.991782,0.473148,15.306080,1.0,0.721569,5.227383,...,0.022173,0.130111,0.022309,0.435980,0.958333,60.307396,0.772092,5.696805,0.186286,0.287986


### 5. Explore results

In [213]:
# Launch interactive explorer (if in Jupyter)
explorer = pheno.create_interactive_explorer()

### Generate plots

In [214]:
PLOTS_PATH = os.path.join(OUTPUT_PATH, "plots")
os.makedirs(PLOTS_PATH, exist_ok=True)

if DATASET_NAME == "NAS":
    color_by = "condition"
    fig=pheno.plot_tsne(color_by=color_by, return_fig=True, render_mode="svg", exclude={"condition":"Block"})
    fig.write_image(os.path.join(PLOTS_PATH, f"{DATASET_NAME}_{MODEL}_noblock_scatterplot.svg"))
elif DATASET_NAME == "Optim":
    color_by = "label"
elif DATASET_NAME == "Zooniverse":
    color_by = "protein"

fig=pheno.plot_tsne(color_by=color_by, return_fig=True, render_mode="svg")
fig.write_image(os.path.join(PLOTS_PATH, f"{DATASET_NAME}_{MODEL}_scatterplot.svg"))